# Day 18: Capstone Mini Project

## Project Goal
This notebook pulls together everything covered so far into one end-to-end project — the same workflow you'd follow on a real job: load data, explore it, clean/prepare it, train a model, then evaluate it honestly on unseen data. The task is binary classification — predicting whether a breast tumor is malignant (cancerous) or benign (not cancerous) from measurements taken from cell images. This kind of workflow generalizes to almost any supervised learning problem, so the goal here isn't just this one dataset — it's practicing the full pipeline so it becomes second nature. Each section below is one stage of that pipeline, and the takeaway is how they fit together, not just what each step does in isolation.</cell id="cell-1">


In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

# as_frame=True returns the dataset as a ready-to-use pandas DataFrame instead of raw numpy arrays
data = load_breast_cancer(as_frame=True)
df = data.frame  # includes both the feature columns and the 'target' label column
print(df.shape)  # (rows, columns) - sanity check on dataset size
print(df.head())

## Explore the Data
Before modeling anything, it's essential to understand what you're working with — skipping this step is one of the most common mistakes beginners make. `df.info()` tells you the data types of each column and whether any values are missing (which would need handling before a model can use them), while checking the target's value counts tells you whether the classes are balanced or skewed. Class balance matters a lot: if 95% of tumors in the data were benign, a model could get 95% accuracy by always predicting "benign" without learning anything useful, so you'd need different evaluation metrics or resampling techniques. The printed output here should show all columns as non-null numeric types (no missing data to clean) and a target count reasonably split between the two classes, though not perfectly even.</cell id="cell-3">


In [ ]:
print(df.info())  # column dtypes + non-null counts, to catch missing values or wrong types
print(df['target'].value_counts())  # how many malignant (0) vs benign (1) cases, to check class balance

## Prepare Features & Target
Every supervised learning problem needs its data split two ways: first into features (X, the inputs the model learns from) and the target (y, what we're trying to predict), and second into training and test sets. The train/test split is critical for honest evaluation — if you evaluate a model on the same data it was trained on, it can look artificially good simply by memorizing the training examples, so a held-out test set simulates seeing genuinely new data. `test_size=0.2` reserves 20% of the rows for testing, and `random_state=42` makes the split reproducible so re-running the notebook gives the same split every time. Nothing is printed here — this cell just creates the four variables (`X_train`, `X_test`, `y_train`, `y_test`) that every later step depends on.</cell id="cell-5">


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['target'])  # all columns except the label are features
y = df['target']  # the column we're trying to predict
# 80/20 split; random_state fixes the shuffle so results are reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Scale Features
Many features in this dataset are measured on very different scales (e.g. "area" can be in the hundreds while "smoothness" is a small decimal), and some ML algorithms (like distance-based or gradient-based models) can be unfairly dominated by features with larger raw magnitudes even if they aren't more important. `StandardScaler` fixes this by transforming each feature to have mean 0 and standard deviation 1, putting every feature on a comparable scale. A crucial detail here: the scaler is *fit* only on the training data (`fit_transform`) and then just *applied* to the test data (`transform`) — this avoids leaking any information from the test set into how the scaling parameters are computed. This cell produces no visible output; it just creates the scaled feature arrays used for training and evaluation.</cell id="cell-7">


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# fit on training data (learns mean/std) and transform it in one step
X_train_scaled = scaler.fit_transform(X_train)
# reuse the SAME scaler (fit on train only) to transform test data - avoids data leakage
X_test_scaled = scaler.transform(X_test)

## Train the Model
With clean, scaled data split into train/test sets, we're ready to fit a model. Random Forest is a strong default choice for classification because it's an ensemble of many decision trees, each trained on a random subset of data and features, whose votes are combined — this tends to generalize well and resist overfitting better than a single decision tree, without much tuning required. `n_estimators=100` sets the number of trees in the forest (more trees generally means a more stable model, at the cost of speed), and `random_state=42` again ensures reproducibility. `.fit()` is where the actual learning happens — the model looks at the training features and labels and learns patterns that separate malignant from benign cases; there's no output to interpret yet since evaluation happens in the next step.</cell id="cell-9">


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 100 trees in the ensemble; each tree votes and the majority class wins
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)  # learn patterns from the scaled training data

## Evaluate the Model
The whole point of the train/test split was to get here: measuring how the model performs on data it has never seen, which is the only honest estimate of how it'll behave in the real world. Accuracy alone (fraction of correct predictions) can be misleading, especially with imbalanced classes, so `classification_report` also gives precision, recall, and F1-score per class. Precision answers "of the cases we predicted malignant, how many actually were?", recall answers "of the actual malignant cases, how many did we catch?" — and in a medical context like this, missing a malignant tumor (low recall) is usually far worse than a false alarm. When reading the output, check both the overall accuracy and the per-class recall/precision — a high accuracy with poor recall on the malignant class would be a real problem despite looking good on the surface.</cell id="cell-11">


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_scaled)  # predictions on the held-out test set
print('Accuracy:', accuracy_score(y_test, y_pred))
# precision, recall, f1-score, and support (sample count) for each class
print(classification_report(y_test, y_pred))